# M3 — Gán nhãn, phân tích, đọc chéo, dataset Hugging Face

Notebook cho cả 4 người. Chạy trên runtime **CPU**. Mỗi phần là một ô riêng, chỉ chạy phần mình cần. Chi tiết định dạng nhãn và các tool: `docs/huong_dan_m3_gan_nhan.md`.

1. Sửa ô cấu hình (đặc biệt `MY_RANGE`), rồi chạy ô *Chuẩn bị* và *Lấy dữ liệu*.
2. **Chọn bài**: chỉ một người chạy; file `selection.csv` nằm trên Drive chung.
3. **Gán nhãn thủ công**: mỗi người gán 5 bài của dải mình. Nhãn được lưu thẳng lên Drive.
4. **Phân tích**, **đọc chéo** (Tiến), **dataset HF** (mỗi người một dataset).

`M3_DRIVE_DIR` phải là thư mục được chia sẻ cho cả nhóm (thêm shortcut vào My Drive nếu cần), để nhãn của 4 người nằm chung một chỗ.

In [ ]:
M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"
M2_SOURCE_RUN_ID = "m2-full-01"
M2_DERIVED_ID = "m2-full-no-empty-roots-01"
M1_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"  # thư mục gốc chứa các run M1 (như notebook QA)
M3_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m3"  # chung cả nhóm: selection, labels, analysis
MY_RANGE = "tien"  # thien / tien / hieu / dang
CODE_REF = "main"
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
LOCAL = "/content/m3-local"

## Chuẩn bị: mount Drive, lấy code

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
repo_dir = Path("/content/CodeRepo-m3")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyarrow",
        "pandas",
        "tqdm",
        "huggingface_hub",
        "datasets",
    ],
    check=True,
)

M2_SOURCE_RUN = Path(M2_DRIVE_DIR) / M2_SOURCE_RUN_ID
M2_RUN = Path(M2_DRIVE_DIR) / "derived" / M2_DERIVED_ID
M3 = Path(M3_DRIVE_DIR)
LABELS = M3 / "labels"
M2_OUT = Path(LOCAL) / M2_DERIVED_ID / "out"  # shard-*/bibtex.parquet
CORPUS = Path(LOCAL) / "corpus"  # <folder>/references.json
M3.mkdir(parents=True, exist_ok=True)

## Lấy dữ liệu về máy Colab

BibTeX lấy từ bộ M2 đã lọc. Manifest của lần parse gốc xác định các checkpoint M1 dùng để lấy references.

Chỉ copy `bibtex.parquet` của M2 (200 shard) và `references.json` của M1 (mỗi bài một file, có kiểm tra SHA-256 theo `INDEX.json`), không restore cả corpus. Nếu mất kết nối giữa chừng, chạy lại ô này: các file đã đúng sẽ được dùng lại.

In [ ]:
import hashlib
import json

from tqdm.auto import tqdm

from src.m3.restore_inputs import restore_bibtex, restore_references


def progress(bar):
    def update(done, total):
        bar.total = total
        bar.update(done - bar.n)

    return update


identity = json.loads((M2_RUN / "identity.json").read_text(encoding="utf-8"))
manifest = (M2_SOURCE_RUN / "M2_MANIFEST.json").read_bytes()
if (
    identity["run_id"] != M2_SOURCE_RUN_ID
    or identity["manifest_sha256"] != hashlib.sha256(manifest).hexdigest()
):
    raise ValueError("Derived M2 output does not match the original M2 manifest")
print("BibTeX:", M2_RUN)
print("M1 checkpoint manifest:", M2_SOURCE_RUN / "M2_MANIFEST.json")

with tqdm(desc="bibtex shards", unit="shard") as bar:
    print(restore_bibtex(M2_RUN, M2_OUT, on_progress=progress(bar)))
with tqdm(desc="references.json", unit="file") as bar:
    print(restore_references(M2_SOURCE_RUN, M1_DRIVE_DIR, CORPUS, on_progress=progress(bar)))

## 1. Chọn 20 bài gán nhãn thủ công (chỉ một người chạy)

Nếu `selection.csv` đã có trên Drive, ô này chỉ in danh sách ra. Muốn chọn lại thì đặt `RESELECT = True`, nhưng **chỉ làm khi chưa ai bắt đầu gán nhãn**.

In [ ]:
import pandas as pd

from src.m3 import select_papers

RESELECT = False
SELECTION = M3 / "selection.csv"
if RESELECT or not SELECTION.exists():
    select_papers.main(
        ["--m2-dir", str(M2_OUT), "--corpus-dir", str(CORPUS), "--out", str(SELECTION)]
    )
selection = pd.read_csv(SELECTION, dtype={"arxiv_id": str})
selection

## 2. Gán nhãn thủ công (mỗi người 5 bài của dải mình)

Ô đầu in tiến độ 5 bài của `MY_RANGE`. Đặt `PAPER` là số thứ tự bài (0–4) rồi chạy ô thứ hai. **Công cụ không điền sẵn; bạn đọc và gõ lựa chọn.**

Phím: `số` = ứng viên đúng · `0` = không có · `a` = hiện tất cả · `/từ` = lọc theo tiêu đề · `b` = quay lại · `q` = lưu và thoát.

Mỗi câu trả lời được ghi ngay lên Drive. Chạy lại ô sẽ tiếp tục từ mục chưa gán.

In [ ]:
from src.m3.labels import label_path, read_matrix

mine = selection[selection.range_owner == MY_RANGE].reset_index(drop=True)
for i, row in mine.iterrows():
    path = label_path(LABELS, "manual", row.arxiv_id)
    done = len(read_matrix(path)[1]) if path.exists() else 0
    print(f"PAPER={i}  {row.arxiv_id}  {done}/{row.n_entries} entries labeled")

In [ ]:
from src.m3 import label_tool

PAPER = 0
label_tool.main(
    [
        "--m2-dir",
        str(M2_OUT),
        "--corpus-dir",
        str(CORPUS),
        "--labels-dir",
        str(LABELS),
        "--arxiv-id",
        mine.arxiv_id[PAPER],
    ]
)

## 3. Phân tích dữ liệu (số liệu cho feature và poster)

Khi đã có `partitions.csv` của Hiếu trong `M3_DRIVE_DIR`, chỉ bài Train được dùng (không rò rỉ từ Valid/Test). Kết quả nằm trong `M3_DRIVE_DIR/analysis/`.

In [ ]:
from src.m3 import analysis

args = [
    "--m2-dir",
    str(M2_OUT),
    "--corpus-dir",
    str(CORPUS),
    "--labels-dir",
    str(LABELS),
    "--out-dir",
    str(M3 / "analysis"),
]
if (M3 / "partitions.csv").exists():
    args += ["--partitions", str(M3 / "partitions.csv")]
else:
    print("partitions.csv chưa có: phân tích trên mọi bài đã gán nhãn (chỉ để xem thử)")
analysis.main(args)

## 4. Đọc chéo nhãn (10 mục mỗi người)

Ô đầu rút mẫu, tạo `crosscheck_sheet.csv`, không chạy lại nếu sheet đã có. Người kiểm tra mở sheet bằng Google Sheets, điền `reviewer_choice` (số ứng viên, `0` nếu không có) và `reviewer`, rồi tải lại về đúng chỗ dưới dạng CSV. **Đừng mở `crosscheck_key.csv` trước khi điền xong.** Ô thứ hai chấm điểm.

In [ ]:
from src.m3 import crosscheck

SHEET, KEY = M3 / "crosscheck_sheet.csv", M3 / "crosscheck_key.csv"
if SHEET.exists():
    print("sheet đã có, không rút lại:", SHEET)
else:
    crosscheck.main(
        [
            "sample",
            "--m2-dir",
            str(M2_OUT),
            "--corpus-dir",
            str(CORPUS),
            "--labels-dir",
            str(LABELS),
            "--sheet",
            str(SHEET),
            "--key",
            str(KEY),
        ]
    )

In [ ]:
crosscheck.main(["score", "--sheet", str(SHEET), "--key", str(KEY)])

## 5. Dataset Hugging Face của dải mình

Cần gói 6 bảng do Hiếu ghi (`papers/ references/ elements/ hierarchy/ bibtex/ predictions/`, mỗi thư mục chứa `*.parquet`). Thêm Colab Secret `HF_TOKEN` (token **write** của tài khoản cá nhân) và bật quyền cho notebook. Không dán token vào ô nào.

Chạy với `DRY_RUN = True` trước để kiểm tra gói và dataset card, rồi đổi thành `False` để upload (public) và tải lại kiểm tra.

In [ ]:
from google.colab import userdata

from src.m3 import hf_dataset

PACKAGE_DIR = M3 / "package" / MY_RANGE
HF_REPO_ID = "<hf-user>/arxiv-nhom1-" + MY_RANGE
DRY_RUN = True

args = [
    "--package-dir",
    str(PACKAGE_DIR),
    "--repo-id",
    HF_REPO_ID,
    "--range",
    MY_RANGE,
    "--labels-dir",
    str(LABELS),
]
if DRY_RUN:
    args.append("--dry-run")
else:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
hf_dataset.main(args)